# Algorithm Forge — Phase 4 Full Pipeline (one-click)

**Ziel:** schließe die AI-evolviert-AI-Loop in einem einzigen Notebook auf gratis Colab T4.

**Was passiert (Run All drücken, ~90 min warten):**
1. Repo clonen, Dependencies installieren
2. N Evolutions-Runs (sort + matmul) mit deinem gewählten LLM-Provider laufen lassen
3. Mutations-Log → Fine-Tuning-Dataset aggregieren
4. Qwen2.5-Coder-7B mit Unsloth auf den Mutations fine-tunen (LoRA, 4-bit)
5. Fine-tuned Modell als GGUF exportieren + zu HF Hub pushen
6. A/B-Vergleich: Base-Qwen vs. Fine-tuned auf 5 frische Seeds
7. Report drucken: hat die Self-Improvement-Loop messbar funktioniert?

**Voraussetzung:** Runtime → T4 GPU + Colab Secrets (`HF_TOKEN`, optional `OPENAI_API_KEY` oder `ANTHROPIC_API_KEY`).

## 1. Konfiguration

Einzige Stelle wo du was ändern musst.

In [ ]:
# --- Modus ---
MODE = "quick"   # "quick" = ~30 min, "thorough" = ~90 min, "deep" = ~3 h

# --- LLM-Provider für die Runs ---
# "openai"  -> braucht OPENAI_API_KEY in Colab Secrets    (~$1-5 fuer thorough)
# "anthropic" -> braucht ANTHROPIC_API_KEY                (~$1-5 fuer thorough, mit prompt-caching billiger)
# "ollama"  -> kostenlos aber langsamer (10x), kein Setup
PROVIDER = "openai"
MODEL_OVERRIDE = None  # None = provider default; sonst z.B. "gpt-4o-mini" oder "claude-sonnet-4-6"

# --- Hugging Face Hub ---
HF_USERNAME = "Beko2210"
HF_DATASET_REPO = f"{HF_USERNAME}/algorithm-forge-mutations-v1"
HF_MODEL_REPO = f"{HF_USERNAME}/algorithm-forge-mutator-qwen-v1"

# --- Repo-Branch ---
REPO_URL = "https://github.com/BEKO2210/Science_game-.git"
BRANCH = "claude/evolution-game-concept-yN3Tn"

# Aus MODE abgeleitet:
if MODE == "quick":
    N_RUNS_PER_BENCH = 3
    GENERATIONS = 15
    FT_EPOCHS = 1
elif MODE == "thorough":
    N_RUNS_PER_BENCH = 8
    GENERATIONS = 30
    FT_EPOCHS = 2
elif MODE == "deep":
    N_RUNS_PER_BENCH = 20
    GENERATIONS = 50
    FT_EPOCHS = 3
else:
    raise ValueError(f"unknown MODE: {MODE}")

print(f"MODE={MODE}  runs/bench={N_RUNS_PER_BENCH}  gens={GENERATIONS}  FT_epochs={FT_EPOCHS}")
print(f"PROVIDER={PROVIDER}  model={MODEL_OVERRIDE or 'default'}")

## 2. Setup: Repo + Dependencies

In [ ]:
import os, subprocess, sys

if not os.path.isdir("Science_game-"):
    subprocess.run(["git", "clone", "--recursive", "-b", BRANCH, REPO_URL], check=True)
os.chdir("Science_game-")
subprocess.run(["git", "pull", "origin", BRANCH], check=True)
print("On commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
# Install uv + project (no torch yet — we'll get the GPU-CUDA torch from Colab base image)
!pip install -qU uv
!uv sync --extra dev --extra api-llm 2>&1 | tail -5

In [ ]:
# API key from Colab Secrets
from google.colab import userdata

if PROVIDER == "openai":
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
elif PROVIDER == "anthropic":
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
elif PROVIDER == "ollama":
    print("Installing + starting Ollama in Colab...")
    !curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
    import subprocess, time
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    !ollama pull qwen2.5-coder:7b

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("Secrets loaded.")

In [ ]:
# Preflight
!uv run science-game doctor --ollama-model qwen2.5-coder:7b 2>&1 | tail -15

## 3. Runs generieren (das ist der lange Schritt)

In [ ]:
import time

_PROVIDER_NAME_MAP = {"openai": "openai", "anthropic": "anthropic", "ollama": "ollama-qwen"}
cli_provider = _PROVIDER_NAME_MAP[PROVIDER]

benchmarks = ["sort", "matmul"]
t0 = time.time()
for bench in benchmarks:
    for seed in range(N_RUNS_PER_BENCH):
        args = [
            "uv", "run", "science-game", "run", bench,
            "--provider", cli_provider,
            "--generations", str(GENERATIONS),
            "--seed", str(seed),
            "--temperature", "0.9",
        ]
        if MODEL_OVERRIDE:
            args.extend(["--model", MODEL_OVERRIDE])
        print(f"\n>>> {bench} seed={seed}")
        proc = subprocess.run(args, capture_output=True, text=True)
        # Last 5 lines so we see the Best-fitness summary.
        for line in proc.stdout.strip().splitlines()[-5:]:
            print("   ", line)
        if proc.returncode != 0:
            print("   STDERR:", proc.stderr.strip().splitlines()[-3:])
print(f"\nAll runs done in {(time.time()-t0)/60:.1f} min")

In [ ]:
# Quick stats over what we just produced
from pathlib import Path
import json

runs = sorted(Path("runs").glob("*/manifest.json"))
print(f"{len(runs)} run(s) on disk")
total_accepted = 0
for r in runs:
    mut_path = r.parent / "mutations.jsonl"
    if not mut_path.exists():
        continue
    rows = [json.loads(l) for l in mut_path.read_text().strip().splitlines() if l.strip()]
    accepted = sum(1 for x in rows if x.get("accepted"))
    total_accepted += accepted
    print(f"  {r.parent.name}: {accepted}/{len(rows)} accepted")
print(f"Total accepted improving mutations: {total_accepted}")

## 4. Mutator-Dataset bauen + zu HF Hub pushen

In [ ]:
!mkdir -p datasets
!uv run science-game build-mutator-dataset \
    --runs-root runs --out datasets/mutator-v1.jsonl \
    --mode sft --min-delta 0.0001 2>&1

In [ ]:
from huggingface_hub import HfApi, create_repo, login

login(token=os.environ["HF_TOKEN"])
create_repo(HF_DATASET_REPO, repo_type="dataset", exist_ok=True, private=False)
HfApi().upload_file(
    path_or_fileobj="datasets/mutator-v1.jsonl",
    path_in_repo="mutator-v1.jsonl",
    repo_id=HF_DATASET_REPO, repo_type="dataset",
)
print(f"Dataset hochgeladen -> https://huggingface.co/datasets/{HF_DATASET_REPO}")

## 5. Fine-tuning mit Unsloth (LoRA, 4-bit, ~30-60 min auf T4)

In [ ]:
# Skip if dataset too small.
import json
n_ds = sum(1 for _ in open("datasets/mutator-v1.jsonl"))
if n_ds < 5:
    raise RuntimeError(
        f"Dataset hat nur {n_ds} examples — zu wenig fuer Fine-Tuning. "
        "Erhoehe N_RUNS_PER_BENCH / GENERATIONS oben oder nimm einen staerkeren Provider "
        "(GPT-4o liefert mehr accepted improvements als qwen)."
    )
print(f"Dataset OK: {n_ds} examples")

In [ ]:
!pip install -qU unsloth
!pip install -qU "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
    max_seq_length=4096,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import standardize_sharegpt

ds = load_dataset(HF_DATASET_REPO, split="train")
ds = standardize_sharegpt(ds)
print(ds)

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    dataset_text_field="text", max_seq_length=4096,
    args=TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=5, num_train_epochs=FT_EPOCHS, learning_rate=2e-4,
        logging_steps=5, optim="adamw_8bit", seed=42,
        output_dir="forge-mutator-out",
    ),
)
trainer.train()

In [ ]:
# Export GGUF + push to HF Hub
create_repo(HF_MODEL_REPO, repo_type="model", exist_ok=True, private=False)
model.push_to_hub_gguf(HF_MODEL_REPO, tokenizer, quantization_method="q4_k_m")
print(f"Modell hochgeladen -> https://huggingface.co/{HF_MODEL_REPO}")

## 6. A/B-Vergleich: Base vs. Fine-tuned

Da Colab unsere Standalone-CLI braucht und das fine-tuned Modell gerade in GGUF-Form vorliegt, registrieren wir es entweder in Ollama (Aufwand) oder vergleichen direkt via Inference-API.

Schnellste Variante: wir laden das fine-tuned Modell via Transformers/Unsloth und schreiben einen kleinen Adapter, der unsere Engine direkt benutzt.

In [ ]:
FastLanguageModel.for_inference(model)

from science_game.benchmarks import get_benchmark
from science_game.engine import Engine, EvolutionConfig
from science_game.llm.base import LLMProvider, MutationRequest, MutationResponse
from science_game.llm.ollama_provider import extract_code


class HFTransformersProvider(LLMProvider):
    name = "hf-finetuned"
    def __init__(self, model, tokenizer, **_):
        self.model = model
        self.tokenizer = tokenizer
    def mutate(self, request):
        prompt = (
            f"Below is the current best program. Produce an improved variant. "
            f"Return ONLY a single fenced Python code block.\n\n"
            f"```python\n{request.parent_code}\n```\n"
        )
        inputs = self.tokenizer(prompt, return_tensors="pt").to("cuda")
        out = self.model.generate(
            **inputs, max_new_tokens=1024,
            temperature=request.temperature, do_sample=True,
        )
        raw = self.tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        return MutationResponse(
            child_code=extract_code(raw), raw_text=raw,
            provider=self.name, model="forge-mutator-v1",
        )

finetuned = HFTransformersProvider(model, tokenizer)
print("Fine-tuned provider bereit.")

In [ ]:
# A/B-Loop selbst: 3 frische Seeds, jeweils Base (via API) vs. Fine-tuned
from science_game.llm import get_provider

AB_BENCH = "sort"
AB_GENERATIONS = 15
AB_SEEDS = [1000, 1001, 1002]

base_provider = get_provider(cli_provider, **({"model": MODEL_OVERRIDE} if MODEL_OVERRIDE else {}))

wins = {"finetuned": 0, "base": 0, "tie": 0}
deltas = []

for s in AB_SEEDS:
    bench = get_benchmark(AB_BENCH)
    base_cfg = EvolutionConfig(benchmark=AB_BENCH, llm_provider=cli_provider,
                               generations=AB_GENERATIONS, seed=s,
                               run_dir=Path(f"runs/ab/seed{s}-base"))
    ft_cfg = EvolutionConfig(benchmark=AB_BENCH, llm_provider="hf-finetuned",
                             generations=AB_GENERATIONS, seed=s,
                             run_dir=Path(f"runs/ab/seed{s}-finetuned"))
    base_best = Engine(base_cfg, bench, base_provider).run()
    ft_best = Engine(ft_cfg, bench, finetuned).run()
    delta = ft_best.result.fitness - base_best.result.fitness
    deltas.append(delta)
    if abs(delta) < 1e-9: wins["tie"] += 1
    elif delta > 0: wins["finetuned"] += 1
    else: wins["base"] += 1
    print(f"seed {s}: base={base_best.result.fitness:.6f}  ft={ft_best.result.fitness:.6f}  delta={delta:+.6f}")

print("\n=== A/B Report ===")
print(f"Wins fine-tuned: {wins['finetuned']}/{len(AB_SEEDS)}")
print(f"Wins base:       {wins['base']}/{len(AB_SEEDS)}")
print(f"Ties:            {wins['tie']}/{len(AB_SEEDS)}")
print(f"Avg delta (ft - base): {sum(deltas)/len(deltas):+.6f}")

## 7. Verdict

Wenn `Avg delta > 0` und Fine-tuned ≥ Base in den meisten Seeds: **die Self-Improvement-Loop hat funktioniert**. Du hast eine eigene KI mit deinen eigenen KI-Mutationsdaten trainiert, und sie ist messbar besser im Code-Mutieren als ihre Base.

Wenn nicht: dann sehen wir uns die Diagnostik an — typische Ursachen sind
- Dataset zu klein (< 20 examples) -> erhoehe MODE auf `thorough` oder `deep`
- Base zu stark (GPT-4o ist sehr gut, da zu gewinnen ist hart) -> nimm `ollama` als Provider, dort ist mehr Headroom
- Fine-tuning zu kurz -> FT_EPOCHS hochsetzen

Alle Artefakte sind auf deinem HF-Hub-Account und in `runs/`.